# 为 Claude 构建自定义技能

学习如何创建、部署和管理自定义技能，用你所在组织的专门知识和工作流扩展 Claude 的能力。

## 目录

1. [引言与环境准备](#introduction)
2. [理解自定义技能架构](#architecture)
3. [示例 1：财务比率计算器](#financial-ratio)
4. [示例 2：公司品牌规范](#brand-guidelines)
5. [示例 3：财务建模套件](#financial-modeling)
6. [技能管理与版本控制](#management)
7. [最佳实践与生产环境建议](#best-practices)
8. [疑难排查](#troubleshooting)

## 1. 引言与环境准备 {#introduction}

### 什么是自定义技能？

**自定义技能**是你创建的专业知识包，用来把所在组织的独特工作流、领域知识和最佳实践教给 Claude。与 Anthropic 预构建的技能（Excel、PowerPoint、PDF）不同，自定义技能让你能够：

- **固化组织知识** - 捕获你团队的具体方法论
- **确保一致性** - 在所有交互中应用同一标准
- **自动化复杂工作流** - 把多步骤流程串联起来
- **保护知识产权** - 让专有方法保持安全

### 关键收益

| 收益 | 说明 |
|---------|-------------|
| **规模化专业知识** | 把专门知识投放到每一次 Claude 交互中 |
| **版本控制** | 跟踪变更，必要时回滚 |
| **可组合性** | 为复杂任务组合多个技能 |
| **隐私** | 你的技能对你所在组织保持私有 |

### 前置条件

开始前，请确认你已具备：
- 已完成[笔记本 1：技能入门](01_skills_introduction.ipynb)
- 具备拥有 Skills 测试版访问权限的 Anthropic API 密钥
- 已安装本地 SDK 的 Python 环境

### 环境准备

我们来配置环境并导入所需的库：

In [ ]:
import os
import sys
from pathlib import Path
from typing import Any

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd().parent))

from anthropic import Anthropic
from anthropic.lib import files_from_dir
from dotenv import load_dotenv

# Import our utilities
from file_utils import (
    download_all_files,
    extract_file_ids,
    print_download_summary,
)

# We'll create skill_utils later in this notebook
# from skill_utils import (
#     create_skill,
#     list_skills,
#     delete_skill,
#     test_skill
# )

# Load environment variables
load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

if not API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY not found. Copy ../.env.example to ../.env and add your API key."
    )

# Initialize client with Skills beta
client = Anthropic(api_key=API_KEY, default_headers={"anthropic-beta": "skills-2025-10-02"})

# Setup directories
SKILLS_DIR = Path.cwd().parent / "custom_skills"
OUTPUT_DIR = Path.cwd().parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")
print(f"✓ Custom skills directory: {SKILLS_DIR}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print("\n📝 Skills beta header configured for skill management")

## 2. 理解自定义技能架构 {#architecture}

### 技能结构

每个自定义技能都遵循以下目录结构：

```
skill_name/
├── SKILL.md           # REQUIRED: Instructions with YAML frontmatter
├── *.md               # Optional: Any additional .md files (documentation, guides)
├── scripts/           # Optional: Executable code
│   ├── process.py
│   └── utils.js
└── resources/         # Optional: Templates, data files
    └── template.xlsx
```

**重要：**
- **SKILL.md 是唯一必需的文件** - 其他一切都是可选的
- **允许多个 .md 文件** - 顶层文件夹中可以有任意数量的 Markdown 文件
- **所有 .md 文件都会被加载** - 不只是 SKILL.md 和 REFERENCE.md，你包含的任何 .md 文件都会加载
- **按需组织** - 用多个 .md 文件来组织复杂文档

📖 阅读我们关于[用技能为智能体装备真实世界能力](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)的工程博客文章

### 技能不只是 Markdown

![Skills Can Include Scripts and Files](../assets/not-just-markdown.png)

技能可以打包多种文件类型：
- **Markdown 文件**：指令和文档（SKILL.md、REFERENCE.md 等）
- **脚本**：Python、JavaScript 或其他可执行代码，用于复杂操作
- **模板**：可定制的预构建文件（Excel 模板、文档模板）
- **资源**：支持性的数据文件、配置或素材

### SKILL.md 的要求

`SKILL.md` 文件必须包含：

1. **YAML 前置元数据**（name：64 字符，description：1024 字符）
   - `name`：小写字母数字加连字符（必需）
   - `description`：简要说明该技能做什么（必需）

2. **指令**（Markdown 格式）
   - 给 Claude 的清晰指引
   - 用法示例
   - 任何约束或规则
   - 建议：控制在 5,000 词元以内

### 额外文档文件

你可以包含多个 Markdown 文件以获得更好的组织：

```
skill_name/
├── SKILL.md           # Main instructions (required)
├── REFERENCE.md       # API reference (optional)
├── EXAMPLES.md        # Usage examples (optional)
├── TROUBLESHOOTING.md # Common issues (optional)
└── CHANGELOG.md       # Version history (optional)
```

技能被加载时，根目录中的所有 `.md` 文件都对 Claude 可用。

### 打包文件示例

![Bundled Files in Skills](../assets/skills-bundled-files.png)

这个示例展示技能如何打包多个文件：
- **SKILL.md**：包含关于配色、字体和章节的主要指令
- **slide-decks.md**：针对特定用例的额外文档
- **脚本与资源**：在技能执行期间可被引用和使用

### 渐进式披露

技能分三个阶段加载，以优化词元用量：

| 阶段 | 内容 | 词元成本 | 加载时机 |
|-------|---------|------------|-------------|
| **1. 元数据** | 名称与描述 | name：64 字符，description：1024 字符 | 始终可见 |
| **2. 指令** | 所有 .md 文件 | 建议少于 5,000 词元 | 相关时 |
| **3. 资源** | 脚本与文件 | 按需 | 执行期间 |

### API 工作流

```python
# 1. Create skill
skill = client.beta.skills.create(
    display_title="My Skill",
    files=files_from_dir("path/to/skill")
)

# 2. Use in messages
response = client.beta.messages.create(
    container={
        "skills": [{
            "type": "custom",
            "skill_id": skill.id,
            "version": "latest"
        }]
    },
    # ... rest of message parameters
)
```

### 最佳实践

关于技能创建的详细指引和最佳实践，参见：
- [Claude 技能最佳实践](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [技能文档](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)

### 创建技能工具函数

我们为技能管理创建辅助函数：

In [ ]:
def create_skill(client: Anthropic, skill_path: str, display_title: str) -> dict[str, Any]:
    """
    Create a new custom skill from a directory.

    Args:
        client: Anthropic client instance
        skill_path: Path to skill directory
        display_title: Human-readable skill name

    Returns:
        Dictionary with skill_id, version, and metadata
    """
    try:
        # Create skill using files_from_dir
        skill = client.beta.skills.create(
            display_title=display_title, files=files_from_dir(skill_path)
        )

        return {
            "success": True,
            "skill_id": skill.id,
            "display_title": skill.display_title,
            "latest_version": skill.latest_version,
            "created_at": skill.created_at,
            "source": skill.source,
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


def list_custom_skills(client: Anthropic) -> list[dict[str, Any]]:
    """
    List all custom skills in the workspace.

    Returns:
        List of skill dictionaries
    """
    try:
        skills_response = client.beta.skills.list(source="custom")

        skills = []
        for skill in skills_response.data:
            skills.append(
                {
                    "skill_id": skill.id,
                    "display_title": skill.display_title,
                    "latest_version": skill.latest_version,
                    "created_at": skill.created_at,
                    "updated_at": skill.updated_at,
                }
            )

        return skills
    except Exception as e:
        print(f"Error listing skills: {e}")
        return []


def delete_skill(client: Anthropic, skill_id: str) -> bool:
    """
    Delete a custom skill and all its versions.

    Args:
        client: Anthropic client
        skill_id: ID of skill to delete

    Returns:
        True if successful, False otherwise
    """
    try:
        # First delete all versions
        versions = client.beta.skills.versions.list(skill_id=skill_id)

        for version in versions.data:
            client.beta.skills.versions.delete(skill_id=skill_id, version=version.version)

        # Then delete the skill itself
        client.beta.skills.delete(skill_id)
        return True

    except Exception as e:
        print(f"Error deleting skill: {e}")
        return False


def test_skill(
    client: Anthropic,
    skill_id: str,
    test_prompt: str,
    model: str = "claude-sonnet-4-6",
) -> Any:
    """
    Test a custom skill with a prompt.

    Args:
        client: Anthropic client
        skill_id: ID of skill to test
        test_prompt: Prompt to test the skill
        model: Model to use for testing

    Returns:
        Response from Claude
    """
    response = client.beta.messages.create(
        model=model,
        max_tokens=4096,
        container={"skills": [{"type": "custom", "skill_id": skill_id, "version": "latest"}]},
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[{"role": "user", "content": test_prompt}],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    return response


print("✓ Skill utility functions defined")
print("  - create_skill()")
print("  - list_custom_skills()")
print("  - delete_skill()")
print("  - test_skill()")

### 检查已有的自定义技能

我们来看看你的工作区中是否已存在自定义技能：

### ⚠️ 重要：开始前清理已有技能

如果你是在重跑本笔记本，可能残留了上一次会话的技能。技能的显示标题不能重复，因此你有三种选择：

1. **删除已有技能**（测试时推荐）- 从零开始的干净做法
2. **使用不同的显示标题** - 在名称中加入时间戳或版本号
3. **用新版本更新已有技能** - 参见[技能管理与版本控制](#management)一节

我们检查已有技能并按需清理：

In [ ]:
# Check for existing skills that might conflict
existing_skills = list_custom_skills(client)
skill_titles_to_create = [
    "Financial Ratio Analyzer",
    "Corporate Brand Guidelines",
    "Financial Modeling Suite",
]
conflicting_skills = []

if existing_skills:
    print(f"Found {len(existing_skills)} existing custom skill(s):")
    for skill in existing_skills:
        print(f"  - {skill['display_title']} (ID: {skill['skill_id']})")
        if skill["display_title"] in skill_titles_to_create:
            conflicting_skills.append(skill)

    if conflicting_skills:
        print(
            f"\n⚠️ Found {len(conflicting_skills)} skill(s) that will conflict with this notebook:"
        )
        for skill in conflicting_skills:
            print(f"  - {skill['display_title']} (ID: {skill['skill_id']})")

        print("\n" + "=" * 70)
        print("To clean up these skills and start fresh, uncomment and run:")
        print("=" * 70)
        print("\n# UNCOMMENT THE LINES BELOW TO DELETE CONFLICTING SKILLS:")
        print("# for skill in conflicting_skills:")
        print("#     if delete_skill(client, skill['skill_id']):")
        print("#         print(f\"✅ Deleted: {skill['display_title']}\")")
        print("#     else:")
        print("#         print(f\"❌ Failed to delete: {skill['display_title']}\")")

        # for skill in conflicting_skills:
        #     if delete_skill(client, skill['skill_id']):
        #         print(f"✅ Deleted: {skill['display_title']}")
        #     else:
        #         print(f"❌ Failed to delete: {skill['display_title']}")
    else:
        print("\n✅ No conflicting skills found. Ready to proceed!")
else:
    print("✅ No existing custom skills found. Ready to create new ones!")

## 3. 示例 1：财务比率计算器 {#financial-ratio}

我们创建第一个自定义技能——一个能够分析公司财务健康状况的财务比率计算器。

### 技能概览

**财务比率计算器**技能将：
- 计算关键财务比率（ROE、P/E、流动比率等）
- 结合行业背景解读比率
- 生成格式化的报告
- 支持多种数据格式（CSV、JSON、文本）

### 上传财务分析技能

现在把我们的财务分析技能上传到 Claude：

In [ ]:
# Upload the Financial Analyzer skill
financial_skill_path = SKILLS_DIR / "analyzing-financial-statements"

if financial_skill_path.exists():
    print("Uploading Financial Analyzer skill...")
    result = create_skill(client, str(financial_skill_path), "Financial Ratio Analyzer")

    if result["success"]:
        financial_skill_id = result["skill_id"]
        print("✅ Skill uploaded successfully!")
        print(f"   Skill ID: {financial_skill_id}")
        print(f"   Version: {result['latest_version']}")
        print(f"   Created: {result['created_at']}")
    else:
        print(f"❌ Upload failed: {result['error']}")
        if "cannot reuse an existing display_title" in str(result["error"]):
            print("\n💡 Solution: A skill with this name already exists.")
            print("   Run the 'Clean Up Existing Skills' cell above to delete it first,")
            print("   or change the display_title to something unique.")
else:
    print(f"⚠️ Skill directory not found: {financial_skill_path}")
    print(
        "Please ensure the custom_skills directory contains the analyzing-financial-statements folder."
    )

### 测试财务分析技能

我们用示例财务数据测试该技能：

In [ ]:
# Test the Financial Analyzer skill
if "financial_skill_id" in locals():
    test_prompt = """
    Calculate financial ratios for this company:

    Income Statement:
    - Revenue: $1,000M
    - EBITDA: $200M
    - Net Income: $120M

    Balance Sheet:
    - Total Assets: $2,000M
    - Current Assets: $500M
    - Current Liabilities: $300M
    - Total Debt: $400M
    - Shareholders Equity: $1,200M

    Market Data:
    - Share Price: $50
    - Shares Outstanding: 100M

    Please calculate key ratios and provide analysis.
    """

    print("Testing Financial Analyzer skill...")
    response = test_skill(client, financial_skill_id, test_prompt)

    # Print response
    for content in response.content:
        if content.type == "text":
            print(content.text)
else:
    print("⚠️ Please upload the Financial Analyzer skill first (run the previous cell)")

## 4. 示例 2：公司品牌规范 {#brand-guidelines}

现在创建一个技能，确保所有文档都遵循公司品牌标准。

### 技能概览

**品牌规范**技能将：
- 应用一致的配色、字体和版式
- 确保标志的位置和使用规范
- 保持专业的语气和信息表达
- 适用于所有文档类型（Excel、PowerPoint、PDF）

In [ ]:
# Upload the Brand Guidelines skill
brand_skill_path = SKILLS_DIR / "applying-brand-guidelines"

if brand_skill_path.exists():
    print("Uploading Brand Guidelines skill...")
    result = create_skill(client, str(brand_skill_path), "Corporate Brand Guidelines")

    if result["success"]:
        brand_skill_id = result["skill_id"]
        print("✅ Skill uploaded successfully!")
        print(f"   Skill ID: {brand_skill_id}")
        print(f"   Version: {result['latest_version']}")
    else:
        print(f"❌ Upload failed: {result['error']}")
        if "cannot reuse an existing display_title" in str(result["error"]):
            print("\n💡 Solution: A skill with this name already exists.")
            print("   Run the 'Clean Up Existing Skills' cell above to delete it first,")
            print("   or change the display_title to something unique.")
else:
    print(f"⚠️ Skill directory not found: {brand_skill_path}")

### 用文档创建测试品牌规范

我们通过创建一份带品牌的 PowerPoint 演示文稿来测试品牌技能：

In [ ]:
# Test Brand Guidelines skill with PowerPoint creation
if "brand_skill_id" in locals():
    # Combine brand skill with Anthropic's pptx skill
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=4096,
        container={
            "skills": [
                {"type": "custom", "skill_id": brand_skill_id, "version": "latest"},
                {"type": "anthropic", "skill_id": "pptx", "version": "latest"},
            ]
        },
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[
            {
                "role": "user",
                "content": """Create a 3-slide PowerPoint presentation following Acme Corporation brand guidelines:

            Slide 1: Title slide for "Q4 2025 Results"
            Slide 2: Revenue Overview with a chart showing Q1-Q4 growth
            Slide 3: Key Achievements (3 bullet points)

            Apply all brand colors, fonts, and formatting standards.
            """,
            }
        ],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    print("Response from Claude:")
    for content in response.content:
        if content.type == "text":
            print(content.text[:500] + "..." if len(content.text) > 500 else content.text)

    # Download generated file
    file_ids = extract_file_ids(response)
    if file_ids:
        results = download_all_files(
            client, response, output_dir=str(OUTPUT_DIR), prefix="branded_"
        )
        print_download_summary(results)
else:
    print("⚠️ Please upload the Brand Guidelines skill first")

## 5. 示例 3：财务建模套件 {#financial-modeling}

我们创建最高级的技能——一套用于估值与风险分析的完整财务建模套件。

### 技能概览

**财务建模套件**技能提供：
- **DCF 估值**：完整的现金流折现模型
- **敏感性分析**：检验各变量对估值的影响
- **蒙特卡洛模拟**：基于概率分布的风险建模
- **情景规划**：最佳/基准/最差情形分析

这展示了一个包含复杂计算和专业级财务建模的多文件技能。

### 上传财务建模套件

首先上传财务建模技能：

In [ ]:
# Upload the Financial Modeling Suite skill
modeling_skill_path = SKILLS_DIR / "creating-financial-models"

if modeling_skill_path.exists():
    print("Uploading Financial Modeling Suite skill...")
    result = create_skill(client, str(modeling_skill_path), "Financial Modeling Suite")

    if result["success"]:
        modeling_skill_id = result["skill_id"]
        print("✅ Skill uploaded successfully!")
        print(f"   Skill ID: {modeling_skill_id}")
        print(f"   Version: {result['latest_version']}")
        print("\nThis skill includes:")
        print("   - DCF valuation model (dcf_model.py)")
        print("   - Sensitivity analysis framework (sensitivity_analysis.py)")
        print("   - Monte Carlo simulation capabilities")
        print("   - Scenario planning tools")
    else:
        print(f"❌ Upload failed: {result['error']}")
else:
    print(f"⚠️ Skill directory not found: {modeling_skill_path}")
    print(
        "Please ensure the custom_skills directory contains the creating-financial-models folder."
    )

### 测试财务建模套件

我们用一次 DCF 估值请求来测试其高级建模能力：

In [ ]:
# Test the Financial Modeling Suite with a DCF valuation
if "modeling_skill_id" in locals():
    dcf_test_prompt = """
    Perform a DCF valuation for TechCorp with the following data:

    Historical Financials (Last 3 Years):
    - Revenue: $500M, $600M, $750M
    - EBITDA Margin: 25%, 27%, 30%
    - CapEx: $50M, $55M, $60M
    - Working Capital: 15% of revenue

    Projections:
    - Revenue growth: 20% for years 1-3, then declining to 5% by year 5
    - EBITDA margin expanding to 35% by year 5
    - Terminal growth rate: 3%

    Market Assumptions:
    - WACC: 10%
    - Tax rate: 25%
    - Current net debt: $200M
    - Shares outstanding: 100M

    Please create a complete DCF model with sensitivity analysis on WACC and terminal growth.
    Generate an Excel file with the full model including:
    1. Revenue projections
    2. Free cash flow calculations
    3. Terminal value
    4. Enterprise value to equity value bridge
    5. Sensitivity table
    """

    print("Testing Financial Modeling Suite with DCF valuation...")
    print("=" * 70)
    print("\n⏱️ Note: Complex financial model generation may take 1-2 minutes.\n")

    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=4096,
        container={
            "skills": [
                {"type": "custom", "skill_id": modeling_skill_id, "version": "latest"},
                {"type": "anthropic", "skill_id": "xlsx", "version": "latest"},
            ]
        },
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[{"role": "user", "content": dcf_test_prompt}],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    # Print Claude's response
    for content in response.content:
        if content.type == "text":
            # Print first 800 characters to keep output manageable
            text = content.text
            if len(text) > 800:
                print(text[:800] + "\n\n[... Output truncated for brevity ...]")
            else:
                print(text)

    # Download the DCF model if generated
    file_ids = extract_file_ids(response)
    if file_ids:
        print("\n" + "=" * 70)
        print("Downloading generated DCF model...")
        results = download_all_files(
            client, response, output_dir=str(OUTPUT_DIR), prefix="dcf_model_"
        )
        print_download_summary(results)
        print("\n💡 Open the Excel file to explore the complete DCF valuation model!")
else:
    print("⚠️ Please upload the Financial Modeling Suite skill first (run the previous cell)")

## 6. 技能管理与版本控制 {#management}

长期管理技能需要理解版本控制、更新和生命周期管理。

### 列出你的技能

获取工作区中所有自定义技能的概览：

In [ ]:
# List all your custom skills
my_skills = list_custom_skills(client)

if my_skills:
    print(f"You have {len(my_skills)} custom skill(s):\n")
    print("=" * 70)
    for i, skill in enumerate(my_skills, 1):
        print(f"\n{i}. {skill['display_title']}")
        print(f"   Skill ID: {skill['skill_id']}")
        print(f"   Current Version: {skill['latest_version']}")
        print(f"   Created: {skill['created_at']}")
        if skill.get("updated_at"):
            print(f"   Last Updated: {skill['updated_at']}")
    print("\n" + "=" * 70)
else:
    print("No custom skills found in your workspace.")

### 创建新版本

技能支持版本控制，以保留历史并支持回滚。我们对财务分析技能做一处增强，并创建一个新版本。

#### 步骤 1：增强财务分析技能

我们将加入**医疗行业**基准数据，让技能更通用。这是一个真实场景：根据用户需求扩展技能的能力。

In [ ]:
# Add healthcare industry benchmarks to the Financial Analyzer
# This demonstrates a realistic skill enhancement scenario

if "financial_skill_id" in locals():
    # Read the current interpret_ratios.py file
    interpret_file_path = SKILLS_DIR / "analyzing-financial-statements" / "interpret_ratios.py"

    with open(interpret_file_path) as f:
        content = f.read()

    # Add healthcare benchmarks after the 'manufacturing' section
    healthcare_benchmarks = """        },
        'healthcare': {
            'current_ratio': {'excellent': 2.3, 'good': 1.8, 'acceptable': 1.4, 'poor': 1.0},
            'debt_to_equity': {'excellent': 0.3, 'good': 0.6, 'acceptable': 1.0, 'poor': 1.8},
            'roe': {'excellent': 0.22, 'good': 0.16, 'acceptable': 0.11, 'poor': 0.07},
            'gross_margin': {'excellent': 0.65, 'good': 0.45, 'acceptable': 0.30, 'poor': 0.20},
            'pe_ratio': {'undervalued': 18, 'fair': 28, 'growth': 40, 'expensive': 55}
        """

    # Find the position after manufacturing section and before the closing brace
    insert_pos = content.find("        }\n    }")  # Find the end of the BENCHMARKS dict

    if insert_pos != -1:
        # Insert the healthcare benchmarks
        new_content = content[:insert_pos] + healthcare_benchmarks + content[insert_pos:]

        # Save the enhanced file
        with open(interpret_file_path, "w") as f:
            f.write(new_content)

        print("✅ Enhanced Financial Analyzer with healthcare industry benchmarks")
        print("\nChanges made:")
        print("  - Added healthcare industry to BENCHMARKS")
        print("  - Includes specific thresholds for:")
        print("    • Current ratio (liquidity)")
        print("    • Debt-to-equity (leverage)")
        print("    • ROE (profitability)")
        print("    • Gross margin")
        print("    • P/E ratio (valuation)")
        print("\n📝 Now we can create a new version of the skill with this enhancement!")
    else:
        print("⚠️ Could not find the correct position to insert healthcare benchmarks")
        print("The file structure may have changed.")
else:
    print("⚠️ Please upload the Financial Analyzer skill first (run cells in Section 3)")

#### 步骤 2：创建新版本

既然已经增强了技能，我们创建一个新版本来跟踪这次变更：

In [ ]:
# Create a new version of the enhanced Financial Analyzer skill
def create_skill_version(client: Anthropic, skill_id: str, skill_path: str):
    """Create a new version of an existing skill."""
    try:
        version = client.beta.skills.versions.create(
            skill_id=skill_id, files=files_from_dir(skill_path)
        )
        return {
            "success": True,
            "version": version.version,
            "created_at": version.created_at,
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


# Create the new version with our healthcare enhancement
if "financial_skill_id" in locals():
    print("Creating new version of Financial Analyzer with healthcare benchmarks...")

    result = create_skill_version(
        client, financial_skill_id, str(SKILLS_DIR / "analyzing-financial-statements")
    )

    if result["success"]:
        print("✅ New version created successfully!")
        print(f"   Version: {result['version']}")
        print(f"   Created: {result['created_at']}")
        print("\n📊 Version History:")
        print("   v1: Original skill with tech, retail, financial, manufacturing")
        print(f"   v{result['version']}: Enhanced with healthcare industry benchmarks")
    else:
        print(f"❌ Version creation failed: {result['error']}")
else:
    print("⚠️ Please run the previous cells to upload the skill and make enhancements first")

#### 步骤 3：测试新版本

我们通过分析一家医疗公司来验证增强是否生效：

In [ ]:
# Test the enhanced skill with healthcare industry data
if "financial_skill_id" in locals():
    healthcare_test_prompt = """
    Analyze this healthcare company using the healthcare industry benchmarks:

    Company: MedTech Solutions (Healthcare Industry)

    Income Statement:
    - Revenue: $800M
    - EBITDA: $320M
    - Net Income: $160M

    Balance Sheet:
    - Total Assets: $1,200M
    - Current Assets: $400M
    - Current Liabilities: $200M
    - Total Debt: $300M
    - Shareholders Equity: $700M

    Market Data:
    - Share Price: $75
    - Shares Outstanding: 50M

    Please calculate key ratios and provide healthcare-specific analysis.
    """

    print("Testing enhanced Financial Analyzer with healthcare company...")
    print("=" * 70)

    response = test_skill(client, financial_skill_id, healthcare_test_prompt, MODEL)

    # Print Claude's analysis
    for content in response.content:
        if content.type == "text":
            # Print first 1000 characters to keep output manageable
            text = content.text
            if len(text) > 1000:
                print(text[:1000] + "\n\n[... Output truncated for brevity ...]")
            else:
                print(text)

    print(
        "\n✅ The skill now recognizes 'healthcare' as an industry and applies specific benchmarks!"
    )
else:
    print("⚠️ Please run the previous cells to create the enhanced version first")

### 清理：管理你的技能

当测试完成或需要清理工作区时，你可以有选择地移除技能。我们回顾一下已创建的内容，并提供清理选项：

In [ ]:
# Comprehensive skill cleanup with detailed reporting
def review_and_cleanup_skills(client, dry_run=True):
    """
    Review all skills and optionally clean up the ones created in this notebook.

    Args:
        client: Anthropic client
        dry_run: If True, only show what would be deleted without actually deleting
    """
    # Get all current skills
    all_skills = list_custom_skills(client)

    # Skills we created in this notebook
    notebook_skill_names = [
        "Financial Ratio Analyzer",
        "Corporate Brand Guidelines",
        "Financial Modeling Suite",
    ]

    # Track skills created by this notebook
    notebook_skills = []
    other_skills = []

    for skill in all_skills:
        if skill["display_title"] in notebook_skill_names:
            notebook_skills.append(skill)
        else:
            other_skills.append(skill)

    print("=" * 70)
    print("SKILL INVENTORY REPORT")
    print("=" * 70)

    print(f"\nTotal custom skills in workspace: {len(all_skills)}")

    if notebook_skills:
        print(f"\n📚 Skills created by this notebook ({len(notebook_skills)}):")
        for skill in notebook_skills:
            print(f"   • {skill['display_title']}")
            print(f"     ID: {skill['skill_id']}")
            print(f"     Version: {skill['latest_version']}")
            print(f"     Created: {skill['created_at']}")
    else:
        print("\n✅ No skills from this notebook found")

    if other_skills:
        print(f"\n🔧 Other skills in workspace ({len(other_skills)}):")
        for skill in other_skills:
            print(f"   • {skill['display_title']} (v{skill['latest_version']})")

    # Cleanup options
    if notebook_skills:
        print("\n" + "=" * 70)
        print("CLEANUP OPTIONS")
        print("=" * 70)

        if dry_run:
            print("\n🔍 DRY RUN MODE - No skills will be deleted")
            print("\nTo delete the notebook skills, uncomment and run:")
            print("-" * 40)
            print("# review_and_cleanup_skills(client, dry_run=False)")
            print("-" * 40)

            print("\nThis would delete:")
            for skill in notebook_skills:
                print(f"   • {skill['display_title']}")
        else:
            print("\n⚠️ DELETION MODE - Skills will be permanently removed")
            print("\nDeleting notebook skills...")

            success_count = 0
            for skill in notebook_skills:
                if delete_skill(client, skill["skill_id"]):
                    print(f"   ✅ Deleted: {skill['display_title']}")
                    success_count += 1
                else:
                    print(f"   ❌ Failed to delete: {skill['display_title']}")

            print(f"\n📊 Cleanup complete: {success_count}/{len(notebook_skills)} skills deleted")

    return {
        "total_skills": len(all_skills),
        "notebook_skills": len(notebook_skills),
        "other_skills": len(other_skills),
        "notebook_skill_ids": [s["skill_id"] for s in notebook_skills],
    }


# Run the review (in dry-run mode by default)
print("Reviewing your custom skills workspace...")
cleanup_summary = review_and_cleanup_skills(client, dry_run=True)

# Store skill IDs for potential cleanup
if cleanup_summary["notebook_skill_ids"]:
    skills_to_cleanup = cleanup_summary["notebook_skill_ids"]
    print(f"\n💡 Tip: {len(skills_to_cleanup)} skill(s) can be cleaned up when you're done testing")

# UNCOMMENT THE LINE BELOW TO ACTUALLY DELETE THE NOTEBOOK SKILLS:
# review_and_cleanup_skills(client, dry_run=False)

## 7. 最佳实践与生产环境建议 {#best-practices}

### 技能设计原则

1. **单一职责**：每个技能应聚焦一个专业领域
2. **清晰的文档**：SKILL.md 应全面而简洁
3. **错误处理**：脚本应妥善处理边界情况
4. **版本控制**：用 Git 跟踪技能变更
5. **测试**：在生产部署之前始终测试技能

### 目录结构最佳实践

```
custom_skills/
├── financial_analyzer/       # Single purpose, clear naming
│   ├── SKILL.md             # Under 5,000 tokens
│   ├── scripts/             # Modular Python/JS files
│   └── tests/               # Unit tests for scripts
├── brand_guidelines/         # Organizational standards
│   ├── SKILL.md
│   ├── REFERENCE.md         # Additional documentation
│   └── assets/              # Logos, templates
```

### 性能优化

| 策略 | 效果 | 实现方式 |
|----------|--------|----------------|
| **精简前置元数据** | 技能发现更快 | name：64 字符，description：1024 字符 |
| **惰性加载** | 词元用量更低 | 仅在需要时引用文件 |
| **技能组合** | 避免重复 | 组合多个技能，而不是做一个巨型技能 |
| **缓存** | 响应更快 | 复用技能容器 |

### 安全注意事项

- **API 密钥**：绝不在技能中硬编码凭据
- **数据隐私**：不要在技能文件中包含敏感数据
- **访问控制**：技能按工作区隔离
- **输入校验**：在脚本中对输入做净化处理
- **审计跟踪**：记录技能使用日志以满足合规要求

## 后续步骤

🎉 **祝贺你！** 你已经学会了如何为 Claude 创建、部署和管理自定义技能。

### 你学到了什么

- ✅ 自定义技能的架构与要求
- ✅ 用 SKILL.md 和 Python 脚本创建技能
- ✅ 通过 API 上传技能
- ✅ 组合自定义技能与 Anthropic 技能
- ✅ 生产部署的最佳实践
- ✅ 排查常见问题

### 继续前进

1. **实验**：为你的用例修改示例技能
2. **构建**：为你所在组织的工作流创建技能
3. **优化**：监控词元用量和性能
4. **分享**：为你的技能编写文档，便于团队协作

### 资源

- [Claude API 文档](https://docs.anthropic.com/en/api/messages)
- [技能文档](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)
- [最佳实践](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [Files API 文档](https://docs.claude.com/en/api/files-content)
- 示例技能仓库（即将推出）

### 可以尝试的技能点子

- 📊 **数据管道**：带校验的 ETL 工作流
- 📝 **文档模板**：合同、提案、报告
- 🔍 **代码审查**：风格指南与最佳实践
- 📈 **分析看板**：KPI 跟踪与可视化
- 🤖 **自动化套件**：重复性任务的工作流

祝你构建技能愉快！🚀